# Atividade 2 — Clusterização Logística | Melhores Compras

**Objetivo:** agrupar entregas malsucedidas por proximidade e calcular o valor atendido por cada centro proposto. As coordenadas do exercício são fictícias.

## 1. Dependências

Execute esta célula na primeira utilização. Se alguma biblioteca for instalada pela primeira vez, reinicie o kernel antes de prosseguir.

In [ ]:
%pip install pandas numpy scikit-learn plotly nbformat

In [ ]:
# Define o limite de threads antes de importar o scikit-learn (Windows).
import os
os.environ["OMP_NUM_THREADS"] = "3"

## 2. Leitura e tratamento da base

O CSV deve estar na mesma pasta do notebook. O arquivo original não é modificado.

In [ ]:
# Bibliotecas utilizadas
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Carrega o CSV original e configura a visualização
ARQ_BASE = pd.read_csv(
    "geolocalizacao_dataset.csv",
    encoding="latin-1", sep=",", decimal=".", low_memory=False
)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 6)

In [ ]:
# Trabalha em uma cópia para preservar a base original.
df = ARQ_BASE.copy()

# Corrige a vírgula decimal nas três colunas.
for coluna in ["latitude", "longitude", "price"]:
    df[coluna] = df[coluna].str.replace(",", ".", regex=False).astype(float)

print(f"Arquivo carregado: {df.shape[0]} registros e {df.shape[1]} colunas")
display(df.head())

In [ ]:
# Verifica tipos, nulos, duplicações e estatísticas antes do modelo.
print("Tipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isna().sum())

print(f"\nLinhas completamente duplicadas: {df.duplicated().sum()}")
print(f"Coordenadas repetidas: {df.duplicated(subset=['latitude', 'longitude']).sum()}")

print("\nResumo estatístico:")

with pd.option_context("display.max_rows", None):
    display(df.describe().round(2))

In [ ]:
# Mantém apenas registros com coordenadas e valores válidos.
validos = df[["latitude", "longitude", "price"]].notna().all(axis=1)
validos &= np.isfinite(df[["latitude", "longitude", "price"]]).all(axis=1)
validos &= df["price"] >= 0

print(f"Linhas inválidas descartadas da análise: {(~validos).sum()}")
df = df.loc[validos].copy()

print(f"Entregas utilizadas: {len(df)}")
print(f"Valor total das entregas: R$ {df['price'].sum():,.2f}")

## 3. Distribuição das entregas

Visualização inicial das coordenadas, com cores correspondentes ao valor de cada entrega.

In [ ]:
# A cor representa o valor financeiro de cada entrega.
scatter = px.scatter(
      df
    , x='longitude'
    , y='latitude'
    , color='price'
    , color_continuous_scale='magma'
    , opacity=0.5
    , width=1000
    , height=600
    , labels={
          'longitude': 'Longitude'
        , 'latitude': 'Latitude'
        , 'price': 'Valor da Entrega (R$)'
    }
    , title='Distribuição Geográfica das Entregas'
    , template='plotly_dark'
    , render_mode='svg'
)

scatter.update_layout(
      plot_bgcolor='#1E1E1E'
    , paper_bgcolor='#121212'
    , font_color='white'
)

scatter.show()

## 4. Seleção das variáveis e avaliação do K

O K-Means considera somente as coordenadas.

In [ ]:
# A proximidade é calculada apenas com latitude e longitude.
X = df[["latitude", "longitude"]]

print(f"Formato das features: {X.shape}")
display(X.head())

In [ ]:
# Testa diferentes valores de K e guarda a inércia e a silhueta.
resultados = []

for k in range(2, 11):
    modelo_teste = KMeans(n_clusters=k, random_state=42, n_init=10)
    grupos_teste = modelo_teste.fit_predict(X)

    resultados.append({
          "K": k
        , "inercia": modelo_teste.inertia_
        , "silhueta": silhouette_score(X, grupos_teste),
    })

metricas = pd.DataFrame(resultados)
with pd.option_context("display.max_rows", None):
    display(metricas.round(4))

In [ ]:
# Cotovelo: queda da inércia; Silhueta: coesão e separação dos grupos.
elbow_plot = px.line(
      metricas
    , x='K'
    , y='inercia'
    , markers=True
    , width=800
    , height=500
    , labels={'K': 'Número de Clusters', 'inercia': 'Inércia'}
    , title='Método do Cotovelo'
    , template='plotly_dark'
)

elbow_plot.update_layout(
      plot_bgcolor='#1E1E1E'
    , paper_bgcolor='#121212'
    , font_color='white'
)
elbow_plot.update_xaxes(dtick=1)

elbow_plot.show()

silhouette_plot = px.line(
      metricas
    , x='K'
    , y='silhueta'
    , markers=True
    , width=800
    , height=500
    , labels={'K': 'Número de Clusters', 'silhueta': 'Coeficiente de Silhueta'}
    , title='Método da Silhueta'
    , template='plotly_dark'
)

silhouette_plot.update_layout(
      plot_bgcolor='#1E1E1E'
    , paper_bgcolor='#121212'
    , font_color='white'
)

silhouette_plot.update_xaxes(dtick=1)

silhouette_plot.show()

## 5. Modelo final e centroides

O K escolhido é aquele com a maior Silhueta entre os valores avaliados. Isso não garante a quantidade economicamente ideal de centros.

In [ ]:
# Seleciona o K com a maior silhueta entre os valores testados.
K_FINAL = int(metricas.loc[metricas["silhueta"].idxmax(), "K"])

modelo = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)

# O +1 deixa os identificadores dos clusters começando em 1.
df["cluster"] = modelo.fit_predict(X) + 1

centroides = modelo.cluster_centers_

print(f"K escolhido: {K_FINAL}")
print(f"Silhueta: {silhouette_score(X, df['cluster']):.4f}")
print(f"Inércia: {modelo.inertia_:.2f}")
print("\nCentroides (latitude, longitude):")
print(centroides.round(4))

In [ ]:
# Visualiza os clusters e marca seus centroides.
df_plot = df.assign(cluster=df['cluster'].astype(str))

cores = {
      '1': '#636EFA'
    , '2': '#EF553B'
    , '3': '#00CC96'
}

scatter_clusters = px.scatter(
      df_plot
    , x='longitude'
    , y='latitude'
    , color='cluster'
    , color_discrete_map=cores
    , width=1000
    , height=800
    , title='Clusters Identificados'
    , render_mode='svg'
)

scatter_clusters.add_trace(
    go.Scatter(
          x=centroides[:, 1]
        , y=centroides[:, 0]
        , mode='markers'
        , marker=dict(color='white', symbol='x', size=20)
        , name='Centroides'
    )
)

scatter_clusters.update_layout(
      plot_bgcolor='#1E1E1E'
    , paper_bgcolor='#121212'
    , font_color='white'
)

scatter_clusters.show()

## 6. Resumo financeiro por cluster

Os valores somados representam o valor das encomendas atendidas, **não** a economia financeira gerada.

In [ ]:
# Resume a quantidade e o valor das entregas atendidas por cluster.
resumo = (
    df.groupby('cluster')
    .agg(
          entregas=('cluster', 'count')
        , valor_total=('price', 'sum')
        , ticket_medio=('price', 'mean')
        , latitude_media=('latitude', 'mean')
        , longitude_media=('longitude', 'mean')
    )
)

resumo['percentual_entregas'] = (resumo['entregas'] / len(df) * 100).round(2)
resumo['percentual_valor'] = (resumo['valor_total'] / df['price'].sum() * 100).round(2)

display(resumo.sort_values(by='entregas', ascending=False).round(2))

print(f"Total de entregas: {resumo['entregas'].sum()}")
print(f"Total do valor das entregas: R$ {resumo['valor_total'].sum():,.2f}")
print(f"Ticket médio geral: R$ {df['price'].mean():,.2f}")

In [ ]:
# Compara o valor total das entregas entre os centros propostos.
bar_plot = px.bar(
      resumo
    , x=resumo.index.astype(str)
    , y='valor_total'
    , text='valor_total'
    , width=800
    , height=800
    , labels={
          'x': 'Cluster / centro de retirada'
        , 'valor_total': 'Valor das entregas (R$)'
    }
    , title='Valor total das entregas por centro proposto'
    , template='plotly_dark'
    , color_discrete_sequence=["#CDF858"]
)

bar_plot.update_traces(
      texttemplate='R$ %{text:,.2f}'
    , textposition='outside'
)

bar_plot.update_layout(
      plot_bgcolor='#1E1E1E'
    , paper_bgcolor='#121212'
    , font_color='white'
    , yaxis=dict(gridcolor='#444444')
    , xaxis_type='category'
    , showlegend=False
)

bar_plot.show()

## 7. Conclusão

Foi aplicado o **K-Means**, algoritmo não supervisionado que agrupa observações pela proximidade de seus centroides. Foram testados valores de **K entre 2 e 10**, utilizando Inércia (Método do Cotovelo) e Índice de Silhueta.

A base contém **653 entregas válidas**, sem registros incompletos ou duplicados. A maior Silhueta entre as opções testadas foi obtida com **K = 2 (0,4703)**. O primeiro grupo reuniu **335 entregas (R$ 20.890,95)**, enquanto o segundo reuniu **318 entregas (R$ 20.292,86)**, totalizando **R$ 41.183,81**.

A análise foi satisfatória como **proposta inicial de duas regiões de atendimento**. Entretanto, as coordenadas são fictícias e o resultado não considera distâncias reais de deslocamento, custos de instalação ou capacidade operacional; portanto, dois clusters não comprovam que dois centros sejam a escolha econômica ideal.